In [ ]:
## EDA
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA as SklearnPCA # Use sklearn's PCA for component calculation
from cuml.decomposition import PCA as CumlPCA # Use cuml's PCA for GPU-accelerated transformation

# 1. Load the dataset
## Adapt the path as needed
df = pd.read_csv('/content/healthcare-dataset-stroke-data.csv')

# 2. Data cleaning: handle missing values
# The bmi column has missing values use the median to fill them
df['bmi'] = df['bmi'].fillna(df['bmi'].median())

# 3. Categorical encoding
# Drop 'id'
df = df.drop(columns=['id'])

# Label encoding for binary features
le = LabelEncoder()
df['gender'] = le.fit_transform(df['gender'])
df['ever_married'] = le.fit_transform(df['ever_married'])
df['Residence_type'] = le.fit_transform(df['Residence_type'])

# Encoding for nominal features:work_type, smoking_status
df = pd.get_dummies(df, columns=['work_type', 'smoking_status'])

# 4. Feature and target split
X = df.drop(columns=['stroke'])
y = df['stroke']

# 5. Scaling the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 6. PCA Implementation
# Reducing features while retaining 95% of the variance

# First, use sklearn PCA to determine the number of components to retain 95% variance
sk_pca_full = SklearnPCA(n_components=None)
sk_pca_full.fit(X_scaled)
n_components_to_keep = np.argmax(np.cumsum(sk_pca_full.explained_variance_ratio_) >= 0.95) + 1

# Now, use cuml PCA with the determined integer number of components
pca = CumlPCA(n_components=n_components_to_keep)
X_pca = pca.fit_transform(X_scaled)

# 7. Splitting into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_pca, y, test_size=0.2, random_state=42)

print(f"Original feature count: {X.shape[1]}")
print(f"Reduced feature count after PCA: {X_pca.shape[1]}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

GPU Model Fitting

In [ ]:
%load_ext cuml.accel

## Decision Tree

Prompt: `using the training data, use GPUs to fit a decision tree model, tune it with hyperparamter tuning, and then using the testing data test the tuned model to give the accuracy, precision, recall, f1 score and confusion matrix of each`

In [ ]:
!pip install cudf-cu12 cuml-cu12 --extra-index-url=https://pypi.nvidia.com

In [ ]:
## MODEL: Decision Tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, accuracy_score, recall_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import time

# 1. Initialize Decision Tree with 'balanced' weights
# This is the single most important line to fix your Recall.
dt_balanced = DecisionTreeClassifier(random_state=42, class_weight='balanced')

# 2. Define Hyperparameter Grid
# We tune depth and splits to prevent overfitting while maintaining sensitivity
param_grid_dt = {
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 10, 20],
    'criterion': ['gini', 'entropy']
}

# 3. Optimize for 'f1' instead of 'accuracy'
# This forces the GridSearch to find a balance between catching strokes and overall correctness.
grid_search_dt = GridSearchCV(dt_balanced, param_grid_dt, cv=5, scoring='f1', verbose=1)

print("Starting Balanced Decision Tree training...")
start_time = time.time()
grid_search_dt.fit(X_train, y_train)
print(f"Training completed in {time.time() - start_time:.2f} seconds.")

# 4. Evaluation
best_dt = grid_search_dt.best_estimator_
y_pred_dt = best_dt.predict(X_test)

print("\n--- Balanced Decision Tree Results ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}")
print(f"Recall (Stroke): {recall_score(y_test, y_pred_dt):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_dt))

# 5. Visualize the change
plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix(y_test, y_pred_dt), annot=True, fmt='d', cmap='Reds',
            xticklabels=['No Stroke', 'Stroke'], yticklabels=['No Stroke', 'Stroke'])
plt.title('Decision Tree Confusion Matrix')
plt.show()

## SVM Model

Prompt: `Using the training data, use GPUs to fit a SVM model and time how long it takes to fit, use GPUs to hyperparameter tune the model and time how long that takes, then using the testing data give the accuracy, precision, recall, f1score, and confusion matrix of the model`

## SVM Model

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, confusion_matrix, classification_report

# 1. Initialize SVM with Balanced Class Weights
# 'balanced' automatically adjusts weights inversely proportional to class frequencies
svm_balanced = SVC(random_state=42, class_weight='balanced', probability=True)

# 2. Define Hyperparameter Grid
param_grid_svm = {
    'C': [0.1, 1, 10],            # Regularization strength
    'kernel': ['rbf', 'linear'],  # RBF is usually better for non-linear medical data
    'gamma': ['scale', 'auto']    # Kernel coefficient
}

# 3. Setup GridSearchCV
# We use 'f1' to balance the trade-off between catching strokes and false alarms
grid_search_svm = GridSearchCV(
    estimator=svm_balanced,
    param_grid=param_grid_svm,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

# 4. Train the Model
print("Starting Balanced SVM training...")
start_time = time.time()
grid_search_svm.fit(X_train, y_train)
end_time = time.time()

# 5. Get the best model
best_svm_model = grid_search_svm.best_estimator_

print(f"Training completed in {end_time - start_time:.2f} seconds.")
print(f"Best Parameters: {grid_search_svm.best_params_}")

# 6. Evaluate on Testing Data
y_pred_svm = best_svm_model.predict(X_test)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred_svm)
recall = recall_score(y_test, y_pred_svm)
precision = precision_score(y_test, y_pred_svm)
f1 = f1_score(y_test, y_pred_svm)
conf_matrix = confusion_matrix(y_test, y_pred_svm)

print("\n--- Balanced SVM Evaluation ---")
print(f"Accuracy: {accuracy:.4f}")
print(f"Recall (Stroke): {recall:.4f}")
print(f"Precision: {precision:.4f}")
print(f"F1-Score: {f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_svm))

# 7. Visualize Confusion Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['No Stroke', 'Stroke'], yticklabels=['No Stroke', 'Stroke'])
plt.title('SVM Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.show()

Starting Balanced SVM training...
Fitting 5 folds for each of 12 candidates, totalling 60 fits


## Random Forest Model

Prompt: `Using the training data, use GPUs to fit a random forest model and time how long it takes to fit, use GPUs to hyperparameter tune the model and time how long that takes, then using the testing data give the accuracy, precision, recall, f1score, and confusion matrix of the model`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, confusion_matrix, classification_report

# 1. Initialize the Random Forest with Balanced Class Weights
# 'balanced' uses the values of y to automatically adjust weights inversely proportional to class frequencies
rf_balanced = RandomForestClassifier(random_state=42, class_weight='balanced')

# 2. Define Hyperparameter Grid
# We focus on depth and estimators to allow the trees enough complexity to find stroke patterns
param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [10, 15, 20],
    'min_samples_split': [2, 5],
    'criterion': ['gini', 'entropy']
}

# 3. Setup GridSearchCV
# We use 'f1' as the scoring metric to ensure we balance Precision and Recall
grid_search_rf = GridSearchCV(
    estimator=rf_balanced,
    param_grid=param_grid_rf,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

# 4. Train the Model
print("Starting Balanced Random Forest training...")
start_time = time.time()
grid_search_rf.fit(X_train, y_train)
end_time = time.time()

# 5. Get the best model
best_rf_model = grid_search_rf.best_estimator_

print(f"Training completed in {end_time - start_time:.2f} seconds.")
print(f"Best Parameters: {grid_search_rf.best_params_}")

# 6. Evaluate on Testing Data
y_pred_rf = best_rf_model.predict(X_test)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred_rf)
recall = recall_score(y_test, y_pred_rf)
precision = precision_score(y_test, y_pred_rf)
f1 = f1_score(y_test, y_pred_rf)
conf_matrix = confusion_matrix(y_test, y_pred_rf)

print("\n--- Balanced Random Forest Evaluation ---")
print(f"Accuracy: {accuracy:.4f}")
print(f"Recall (Stroke): {recall:.4f}")
print(f"Precision: {precision:.4f}")
print(f"F1-Score: {f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

# 7. Visualize Confusion Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Oranges', cbar=False,
            xticklabels=['No Stroke', 'Stroke'], yticklabels=['No Stroke', 'Stroke'])
plt.title('Random Forest Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, confusion_matrix, classification_report

# 1. Initialize the Random Forest with Balanced Class Weights
rf_balanced = RandomForestClassifier(random_state=42, class_weight='balanced')

# 2. Define Hyperparameter Grid
param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [10, 15, 20],
    'min_samples_split': [2, 5],
    'criterion': ['gini', 'entropy']
}

# 3. Setup GridSearchCV (Optimizing for F1)
grid_search_rf = GridSearchCV(
    estimator=rf_balanced,
    param_grid=param_grid_rf,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

# 4. Train the Model
print("Starting Balanced Random Forest training...")
start_time = time.time()
grid_search_rf.fit(X_train, y_train)
end_time = time.time()

# 5. Get the best model
best_rf_model = grid_search_rf.best_estimator_
print(f"Training completed in {end_time - start_time:.2f} seconds.")

# 6. Evaluate on Testing Data with ADJUSTED THRESHOLD (0.2)
# Instead of .predict(), we use .predict_proba()
y_probs_rf = best_rf_model.predict_proba(X_test)[:, 1]
custom_threshold = 0.2
y_pred_adj = (y_probs_rf >= custom_threshold).astype(int)

# Calculate metrics for the adjusted threshold
accuracy = accuracy_score(y_test, y_pred_adj)
recall = recall_score(y_test, y_pred_adj)
precision = precision_score(y_test, y_pred_adj)
f1 = f1_score(y_test, y_pred_adj)
conf_matrix = confusion_matrix(y_test, y_pred_adj)

print(f"\n--- Random Forest Evaluation (Threshold: {custom_threshold}) ---")
print(f"Accuracy: {accuracy:.4f}")
print(f"Recall (Stroke): {recall:.4f}")
print(f"Precision: {precision:.4f}")
print(f"F1-Score: {f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_adj))

# 7. Visualize Confusion Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Oranges', cbar=False,
            xticklabels=['No Stroke', 'Stroke'], yticklabels=['No Stroke', 'Stroke'])
plt.title(f'Random Forest Confusion Matrix (Threshold: {custom_threshold})')
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.show()

## Naive Bayes Model

Prompt: `Using the training data, use GPUs to fit a naive bayes model and time how long it takes to fit, use GPUs to hyperparameter tune the model and time how long that takes, then using the testing data give the accuracy, precision, recall, f1score, and confusion matrix of the model
`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.naive_bayes import GaussianNB # Fixed import path
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, confusion_matrix, classification_report

# 1. Initialize Naive Bayes
# We manually set priors to give 'Stroke' (class 1) more weight.
# This forces the model to prioritize detecting the minority class.
nb_model = GaussianNB(priors=[0.4, 0.6])

# 2. Train the Model
print("Starting Balanced Naive Bayes training...")
start_time = time.time()
nb_model.fit(X_train, y_train)
end_time = time.time()

print(f"Training completed in {end_time - start_time:.4f} seconds.")

# 3. Evaluate
y_pred_nb = nb_model.predict(X_test)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred_nb)
recall = recall_score(y_test, y_pred_nb)
precision = precision_score(y_test, y_pred_nb, zero_division=0)
f1 = f1_score(y_test, y_pred_nb, zero_division=0)
conf_matrix = confusion_matrix(y_test, y_pred_nb)

print("\n--- Balanced Naive Bayes Evaluation ---")
print(f"Accuracy: {accuracy:.4f}")
print(f"Recall (Stroke): {recall:.4f}")
print(f"Precision: {precision:.4f}")
print(f"F1-Score: {f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_nb, zero_division=0))

# 4. Visualize Confusion Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='YlGnBu', cbar=False,
            xticklabels=['No Stroke', 'Stroke'], yticklabels=['No Stroke', 'Stroke'])
plt.title('Naive Bayes Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.show()

## KNN Model

Prompt: `Using the training data, use GPUs to fit a knn model and time how long it takes to fit, use GPUs to hyperparameter tune the model and time how long that takes, then using the testing data give the accuracy, precision, recall, f1score, and confusion matrix of the model`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, confusion_matrix, classification_report

# 1. Initialize KNN with Distance Weighting
# 'weights=distance' ensures closer neighbors have more influence than further ones
knn_model = KNeighborsClassifier(n_neighbors=5, weights='distance')

# 2. Train the Model
print("Starting Weighted KNN training...")
start_time = time.time()
knn_model.fit(X_train, y_train)
end_time = time.time()

print(f"Training completed in {end_time - start_time:.2f} seconds.")

# 3. Evaluate with ADJUSTED THRESHOLD (0.2)
# KNN's predict_proba represents the fraction of neighbors belonging to each class
y_probs_knn = knn_model.predict_proba(X_test)[:, 1]
custom_threshold = 0.2
y_pred_knn = (y_probs_knn >= custom_threshold).astype(int)

# 4. Calculate metrics
accuracy = accuracy_score(y_test, y_pred_knn)
recall = recall_score(y_test, y_pred_knn)
precision = precision_score(y_test, y_pred_knn)
f1 = f1_score(y_test, y_pred_knn)
conf_matrix = confusion_matrix(y_test, y_pred_knn)

print(f"\n--- KNN Evaluation (Threshold: {custom_threshold}) ---")
print(f"Accuracy: {accuracy:.4f}")
print(f"Recall (Stroke): {recall:.4f}")
print(f"Precision: {precision:.4f}")
print(f"F1-Score: {f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_knn))

# 5. Visualize Confusion Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Purples', cbar=False,
            xticklabels=['No Stroke', 'Stroke'], yticklabels=['No Stroke', 'Stroke'])
plt.title(f'KNN Confusion Matrix (Threshold: {custom_threshold})')
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.show()

Create a comparison table and plot of all models to provide better visuals.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. Create a list of the models and their corresponding prediction variables
# We use the specific variables generated in our optimized runs
model_map = {
    'Decision Tree': y_pred_dt,
    'SVM': y_pred_svm,
    'Random Forest': y_pred_adj,  # Using the 0.2 threshold version
    'Naive Bayes': y_pred_nb,
    'KNN': y_pred_knn              # Using the adjusted version
}

results_list = []

# 2. Dynamically calculate metrics for each model
for name, preds in model_map.items():
    results_list.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds, zero_division=0),
        'Recall': recall_score(y_test, preds),
        'F1-Score': f1_score(y_test, preds, zero_division=0)
    })

# 3. Convert to DataFrame
df_results = pd.DataFrame(results_list)

# 4. Display the Table (Sorted by Recall to highlight clinical success)
print("### Final Model Comparison ###")
print(df_results.sort_values(by='Recall', ascending=False).to_string(index=False))

# 5. Visualization
# Melting the dataframe makes it compatible with seaborn's 'hue' parameter
df_melted = df_results.melt(id_vars='Model', var_name='Metric', value_name='Score')

plt.figure(figsize=(12, 6))
sns.barplot(data=df_melted, x='Model', y='Score', hue='Metric', palette='viridis')

plt.title('Model Performance Comparison (Optimized Metrics)')
plt.ylim(0, 1.1)
plt.ylabel('Score (0.0 - 1.0)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
## --- FINAL DETERMINATION OF SIGNIFICANT PREDICTORS ---
import pandas as pd
import numpy as np

# 1. Get the raw importance weights from your GPU Random Forest
importances = rf_gpu.feature_importances_

# 2. Get the actual number of features the model saw
num_features_model_saw = len(importances)

# 3. Use the columns from X_train (even if it's a numpy array, we'll map it back)
# This ensures we match the correct label to the correct weight
if hasattr(X_train, 'columns'):
    current_feature_names = X_train.columns
else:
    # If X_train is a numpy array, we use the names from the original dataframe
    # but only up to the count the model actually used.
    current_feature_names = feature_names[:num_features_model_saw]

# 4. Create the ranking DataFrame
predictor_ranking = pd.DataFrame({
    'Clinical Predictor': current_feature_names,
    'Statistical Weight': importances
})

# 5. Sort by most significant
predictor_ranking = predictor_ranking.sort_values(by='Statistical Weight', ascending=False).reset_index(drop=True)

# 6. Display the Final Results
print("==============================================")
print("   DETERMINED SIGNIFICANT RISK FACTORS        ")
print("==============================================")
print(predictor_ranking)
print("==============================================")

# This addresses your "Highest Statistical Significance" goal
top_3 = predictor_ranking.head(3)['Clinical Predictor'].tolist()
print(f"Top 3 Significant Predictors identified: {', '.join(top_3)}")

In [ ]:
import pandas as pd

# 1. Collect your results (assuming you've stored your scores in variables)
# If your variables have different names, just swap them here
models = ['Random Forest', 'SVM', 'KNN', 'Decision Tree', 'Naive Bayes']

# Replace these placeholders with actual score variables from code
accuracies = [rf_acc, svm_acc, knn_acc, dt_acc, nb_acc]
recalls = [rf_recall, svm_recall, knn_recall, dt_recall, nb_recall]
f1_scores = [rf_f1, svm_f1, knn_f1, dt_f1, nb_f1]

# 2. Create the Results DataFrame
final_results = pd.DataFrame({
    'Model': models,
    'Accuracy': accuracies,
    'Recall': recalls,
    'F1-Score': f1_scores
})

# 3. Sort by Recall (since catching strokes is our priority!)
final_results = final_results.sort_values(by='Recall', ascending=False)

# 4. Display the table
print("--- Final Model Performance Comparison ---")
display(final_results)

# 5. Export to CSV for presentation slides
final_results.to_csv('model_results_comparison.csv', index=False)